In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.

# filename: fix_sample_data_dates.py
"""
TICKET-1001 Practice Fix
Problem: Sample data has 2023 dates but current year is 2026
Solution: Regenerate data with dates relative to TODAY
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, rand, date_add,
    lit, current_date, expr,
    when, year
)

spark = SparkSession.builder \
    .appName("Fix-SampleData-Dates") \
    .getOrCreate()

print("🔄 Regenerating sample data with current dates...")

# ── Regenerate with dates from LAST 2 YEARS from TODAY ────────────
# This ensures "last 90 days" filter actually finds data

df_base = spark.range(0, 2_000_000).toDF("sale_id")

df_fixed = df_base \
    .withColumn(
        "sale_date",
        # Dates: today going back 730 days (2 years)
        # Ensures last 30/60/90 days ALWAYS has data
        date_add(
            current_date(),
            (-(rand() * 730)).cast("int")   # negative = going back in time
        )
    ) \
    .withColumn(
        "city",
        when(col("sale_id") % 12 == 0,  lit("New York"))
        .when(col("sale_id") % 12 == 1,  lit("Los Angeles"))
        .when(col("sale_id") % 12 == 2,  lit("Chicago"))
        .when(col("sale_id") % 12 == 3,  lit("Houston"))
        .when(col("sale_id") % 12 == 4,  lit("Phoenix"))
        .when(col("sale_id") % 12 == 5,  lit("Philadelphia"))
        .when(col("sale_id") % 12 == 6,  lit("San Antonio"))
        .when(col("sale_id") % 12 == 7,  lit("San Diego"))
        .when(col("sale_id") % 12 == 8,  lit("Dallas"))
        .when(col("sale_id") % 12 == 9,  lit("San Jose"))
        .when(col("sale_id") % 12 == 10, lit("Austin"))
        .otherwise(lit("Jacksonville"))
    ) \
    .withColumn(
        "product_category",
        when(col("sale_id") % 9 == 0, lit("Electronics"))
        .when(col("sale_id") % 9 == 1, lit("Clothing"))
        .when(col("sale_id") % 9 == 2, lit("Furniture"))
        .when(col("sale_id") % 9 == 3, lit("Sports"))
        .when(col("sale_id") % 9 == 4, lit("Toys"))
        .when(col("sale_id") % 9 == 5, lit("Books"))
        .when(col("sale_id") % 9 == 6, lit("Grocery"))
        .when(col("sale_id") % 9 == 7, lit("Beauty"))
        .otherwise(lit("Automotive"))
    ) \
    .withColumn("amount",       (rand() * 990 + 10).cast("double")) \
    .withColumn("quantity",     (rand() * 9 + 1).cast("int")) \
    .withColumn("customer_id",  (rand() * 50000 + 1).cast("int")) \
    .withColumn(
        "payment_method",
        when(col("sale_id") % 5 == 0, lit("Credit Card"))
        .when(col("sale_id") % 5 == 1, lit("Debit Card"))
        .when(col("sale_id") % 5 == 2, lit("PayPal"))
        .when(col("sale_id") % 5 == 3, lit("Gift Card"))
        .otherwise(lit("Cash"))
    ) \
    .withColumn("year",  col("sale_date").cast("string").substr(1, 4))

# ── Write in 50 small batches (recreate the problem) ─────────────
print("📝 Writing 50 small batches to recreate fragmentation...")

# Overwrite existing table
df_fixed.limit(40_000).write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year") \
    .saveAsTable("retailmart.sales_transactions")

# 49 more small batches
for batch in range(1, 50):
    offset = batch * 40_000
    df_fixed.limit(offset + 40_000) \
            .exceptAll(df_fixed.limit(offset)) \
            .write \
            .format("delta") \
            .mode("append") \
            .partitionBy("year") \
            .saveAsTable("retailmart.sales_transactions")

    if batch % 10 == 0:
        print(f"  Batch {batch}/49 done...")

# ── Verify Fix ────────────────────────────────────────────────────
print("\n✅ Data regenerated! Verifying date ranges...")

spark.sql("""
    SELECT
        MIN(sale_date) AS earliest_date,
        MAX(sale_date) AS latest_date,
        COUNT(*)       AS total_rows
    FROM retailmart.sales_transactions
""").show()

# Verify last 90 days has data now
spark.sql("""
    SELECT COUNT(*) AS rows_last_90_days
    FROM retailmart.sales_transactions
    WHERE sale_date >= date_sub(current_date(), 90)
""").show()

print("🎯 Now re-run TICKET-1001-investigation.py to get REAL results!")

